# Logistic Regression - Grid Search & Random Search

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import copy
import time
import itertools
import random
from data_loader import load_clinc

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = False
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
train_features = vectorizer.fit_transform(train_texts).toarray()
val_features = vectorizer.transform(val_texts).toarray()
test_features = vectorizer.transform(test_texts).toarray()
print(f"TF-IDF dim: {train_features.shape[1]}")

cuda


15250 / 3100 / 5500, 151 classes (full)
TF-IDF dim: 5000


In [2]:
class TextDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class LogisticRegression(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
PATIENCE = 5
MAX_EPOCHS = 50


def evaluate_hyperparams(params):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                all_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    return best_f1, epoch + 1

## Grid Search

In [4]:
grid = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3],
    'batch_size': [32, 64, 128, 256]
}

keys = list(grid.keys())
combinations = list(itertools.product(*grid.values()))
BUDGET = len(combinations)
print(f"Total combinations (budget): {BUDGET}")

grid_results = []
best_f1, best_params = 0, None
start_time = time.time()

for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    eval_start = time.time()
    f1, epochs = evaluate_hyperparams(params)
    eval_time = time.time() - eval_start
    grid_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs,
                         "eval_time": round(eval_time, 2)})

    if f1 > best_f1:
        best_f1, best_params = f1, params

    if (i + 1) % 16 == 0 or i == 0 or i == len(combinations) - 1:
        print(f"{i+1:3d}/{BUDGET}  f1={f1:.4f}  {params}")

grid_time = time.time() - start_time
print(f"\nGrid Search done. Best F1: {best_f1:.4f}, Time: {grid_time:.1f}s")
print(f"Best params: {best_params}")

Total combinations (budget): 96
  1/96  f1=0.8451  {'learning_rate': 0.0001, 'weight_decay': 0, 'batch_size': 32}
 16/96  f1=0.5914  {'learning_rate': 0.0001, 'weight_decay': 0.001, 'batch_size': 256}
 32/96  f1=0.6018  {'learning_rate': 0.0005, 'weight_decay': 0.001, 'batch_size': 256}
 48/96  f1=0.5915  {'learning_rate': 0.001, 'weight_decay': 0.001, 'batch_size': 256}
 64/96  f1=0.6325  {'learning_rate': 0.005, 'weight_decay': 0.001, 'batch_size': 256}
 80/96  f1=0.6303  {'learning_rate': 0.01, 'weight_decay': 0.001, 'batch_size': 256}
 96/96  f1=0.6455  {'learning_rate': 0.05, 'weight_decay': 0.001, 'batch_size': 256}

Grid Search done. Best F1: 0.8964, Time: 3876.5s
Best params: {'learning_rate': 0.05, 'weight_decay': 1e-05, 'batch_size': 128}


## Random Search

In [5]:
search_space = {
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05],
    'weight_decay': [0, 1e-5, 1e-4, 1e-3],
    'batch_size': [32, 64, 128, 256]
}

SEEDS = [42, 123, 456, 789, 1024]
all_random_runs = []

for run_idx, seed in enumerate(SEEDS):
    random.seed(seed)
    random_results = []
    best_f1_rand, best_params_rand = 0, None
    start_time = time.time()

    for i in range(BUDGET):
        params = {k: random.choice(v) for k, v in search_space.items()}
        eval_start = time.time()
        f1, epochs = evaluate_hyperparams(params)
        eval_time = time.time() - eval_start
        random_results.append({"params": params, "val_f1": round(f1, 4), "epochs": epochs,
                               "eval_time": round(eval_time, 2)})

        if f1 > best_f1_rand:
            best_f1_rand, best_params_rand = f1, params

    random_time = time.time() - start_time
    all_random_runs.append({
        "seed": seed,
        "best_f1": best_f1_rand,
        "best_params": best_params_rand,
        "time": random_time,
        "all_results": random_results
    })
    print(f"Run {run_idx+1}/5 (seed={seed}): Best F1={best_f1_rand:.4f}, Time={random_time:.1f}s")

random_val_f1s = [r['best_f1'] for r in all_random_runs]
print(f"\nRandom Search val F1 (5 runs): {np.mean(random_val_f1s):.4f} ± {np.std(random_val_f1s):.4f}")

Run 1/5 (seed=42): Best F1=0.8964, Time=4324.4s
Run 2/5 (seed=123): Best F1=0.8958, Time=3630.7s
Run 3/5 (seed=456): Best F1=0.8964, Time=3242.6s
Run 4/5 (seed=789): Best F1=0.8964, Time=3209.0s
Run 5/5 (seed=1024): Best F1=0.8964, Time=2939.9s

Random Search val F1 (5 runs): 0.8963 ± 0.0003


## Test Evaluation

In [6]:
def train_and_test(params, label):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(TextDataset(train_features, train_labels), batch_size=params['batch_size'], shuffle=True)
    vl_loader = DataLoader(TextDataset(val_features, val_labels), batch_size=params['batch_size'], shuffle=False)
    te_loader = DataLoader(TextDataset(test_features, test_labels), batch_size=params['batch_size'], shuffle=False)

    model = LogisticRegression(train_features.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])

    best_f1, best_state, patience_counter = 0, copy.deepcopy(model.state_dict()), 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for features, labels in tr_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        vl_preds = []
        with torch.no_grad():
            for features, labels in vl_loader:
                vl_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
        if vl_f1 > best_f1:
            best_f1, best_state, patience_counter = vl_f1, copy.deepcopy(model.state_dict()), 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    model.load_state_dict(best_state)
    model.eval()
    te_preds = []
    with torch.no_grad():
        for features, labels in te_loader:
            te_preds.extend(model(features.to(device)).argmax(1).cpu().numpy())

    acc = accuracy_score(test_labels, te_preds)
    p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
    r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
    f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

    print(f"{label}:")
    print(f"    Accuracy:   {acc:.4f}")
    print(f"    Precision:  {p:.4f}")
    print(f"    Recall:     {r:.4f}")
    print(f"    F1 macro:   {f1_m:.4f}")
    print(f"    F1 weighted:{f1_w:.4f}")

    return {
        "metrics": {
            "accuracy": round(acc, 4), "macro_precision": round(p, 4),
            "macro_recall": round(r, 4), "macro_f1": round(f1_m, 4), "weighted_f1": round(f1_w, 4)
        },
        "predictions": [int(x) for x in te_preds]
    }


grid_result = train_and_test(best_params, "Grid Search")
grid_metrics = grid_result['metrics']
grid_preds = grid_result['predictions']

print()

random_test_results = []
for run in all_random_runs:
    result = train_and_test(run['best_params'], f"Random (seed={run['seed']})")
    random_test_results.append(result)

random_test_f1s = [r['metrics']['macro_f1'] for r in random_test_results]
print(f"\nRandom test F1 (5 runs): {np.mean(random_test_f1s):.4f} ± {np.std(random_test_f1s):.4f}")

rand_metrics = random_test_results[0]['metrics']
best_params_rand = all_random_runs[0]['best_params']

Grid Search:
    Accuracy:   0.8207
    Precision:  0.8310
    Recall:     0.8978
    F1 macro:   0.8576
    F1 weighted:0.8127

Random (seed=42):
    Accuracy:   0.8207
    Precision:  0.8310
    Recall:     0.8978
    F1 macro:   0.8576
    F1 weighted:0.8127
Random (seed=123):
    Accuracy:   0.8204
    Precision:  0.8305
    Recall:     0.8945
    F1 macro:   0.8561
    F1 weighted:0.8127
Random (seed=456):
    Accuracy:   0.8207
    Precision:  0.8310
    Recall:     0.8978
    F1 macro:   0.8576
    F1 weighted:0.8127
Random (seed=789):
    Accuracy:   0.8207
    Precision:  0.8310
    Recall:     0.8978
    F1 macro:   0.8576
    F1 weighted:0.8127
Random (seed=1024):
    Accuracy:   0.8207
    Precision:  0.8310
    Recall:     0.8978
    F1 macro:   0.8576
    F1 weighted:0.8127

Random test F1 (5 runs): 0.8573 ± 0.0006


## Save Results

In [7]:
gs_out = {
    "model_type": "LogisticRegression",
    "optimization": "grid_search",
    "best_hyperparameters": best_params,
    "total_evaluations": len(combinations),
    "search_time_seconds": round(grid_time, 2),
    "test_metrics": grid_metrics,
    "predictions": grid_preds,
    "all_results": grid_results
}
with open('results/results_lr_grid.json', 'w') as f:
    json.dump(gs_out, f, indent=4, default=str)

rs_out = {
    "model_type": "LogisticRegression",
    "optimization": "random_search",
    "n_runs": 5,
    "seeds": SEEDS,
    "val_f1_mean": round(np.mean(random_val_f1s), 4),
    "val_f1_std": round(np.std(random_val_f1s), 4),
    "test_f1_mean": round(np.mean(random_test_f1s), 4),
    "test_f1_std": round(np.std(random_test_f1s), 4),
    "total_evaluations_per_run": BUDGET,
    "runs": [{
        "seed": r['seed'],
        "best_hyperparameters": r['best_params'],
        "val_f1": round(r['best_f1'], 4),
        "search_time_seconds": round(r['time'], 2),
        "test_metrics": random_test_results[i]['metrics'],
        "predictions": random_test_results[i]['predictions'],
        "all_results": r['all_results']
    } for i, r in enumerate(all_random_runs)]
}
with open('results/results_lr_random.json', 'w') as f:
    json.dump(rs_out, f, indent=4, default=str)

print("Saved: results/results_lr_grid.json, results/results_lr_random.json")

Saved: results/results_lr_grid.json, results/results_lr_random.json
